# 𓂀 Hieroglyphics Detection & Translation

**Graduation Project — Artificial Intelligence**

This notebook implements an end-to-end pipeline for:
1. **Symbol Detection** — Detecting hieroglyphic symbols in images using OpenCV
2. **Symbol Classification** — Classifying each symbol using a Siamese Network (SqueezeNet)
3. **Meaning Lookup** — Retrieving the meaning of each symbol from the Gardiner list (754 symbols)
4. **Text Translation** — Translating the full hieroglyphic text column by column using Gemini AI

**My Contribution:** Built and trained the Siamese Network for symbol classification + full pipeline integration

---
**Dataset:** [Hieroglyphs Dataset on Kaggle](https://www.kaggle.com/datasets/ayatollahelkolally/hieroglyphs-dataset)  
**Model:** Siamese Network with SqueezeNet backbone  
**Translation:** Google Gemini AI

## 1. Setup & Installation

In [ ]:
# Install required libraries
!pip install google-generativeai -q
!git clone https://github.com/MalakSadek/Dua-Khety.git -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Imports
import cv2
import torch
import pickle
import numpy as np
import re
import matplotlib.pyplot as plt
import google.generativeai as genai
from PIL import Image
from scipy.spatial.distance import cosine
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Model Architecture

In [ ]:
class EmbeddingNet(nn.Module):
    """
    Feature extractor using SqueezeNet backbone.
    Extracts 512-dimensional embeddings from hieroglyphic symbol images.
    """
    def __init__(self):
        super(EmbeddingNet, self).__init__()
        self.squeezenet = models.squeezenet1_1(weights=None)
        self.squeezenet.classifier = nn.Sequential()
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.squeezenet.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x


class SiameseNet(nn.Module):
    """
    Siamese Network for hieroglyphic symbol similarity learning.
    Takes two symbol images and outputs a similarity score (0-1).
    Architecture: EmbeddingNet + FC layers (512 -> 256 -> 128 -> 1)
    """
    def __init__(self, embedding_net):
        super(SiameseNet, self).__init__()
        self.embedding_net = embedding_net
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1)
        )

    def forward(self, x1, x2):
        out1 = self.embedding_net(x1)
        out2 = self.embedding_net(x2)
        diff = torch.abs(out1 - out2)
        out = self.fc(diff)
        return torch.sigmoid(out)


print('Model architecture defined successfully!')

## 3. Load Pretrained Model

In [ ]:
# Load the trained Siamese Network
# Download model from: [add your Google Drive link here]
MODEL_PATH = '/content/siamese_squeezenet.pth'

embedding_net = EmbeddingNet()
model = SiameseNet(embedding_net)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

print('✅ Model loaded successfully!')

# Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

## 4. Generate Reference Embeddings

In [ ]:
# Generate embeddings for all symbols in the dataset
# This creates a reference database for classification
DATASET_PATH = '/content/drive/MyDrive/datasets/Dataset_organized/organized_dataset_by_class_1'

reference_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
loader = DataLoader(reference_dataset, batch_size=32, shuffle=False)

embeddings_data = []
print('Generating embeddings for reference dataset...')

with torch.no_grad():
    for imgs, labels in loader:
        imgs = imgs.to(device)
        embs = model.embedding_net(imgs).cpu().numpy()
        for emb, label in zip(embs, labels):
            embeddings_data.append({
                'embedding': emb,
                'class_name': reference_dataset.classes[label]
            })

# Save embeddings
with open('saved_embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings_data, f)

print(f'✅ Done! Total reference symbols: {len(embeddings_data)}')
print(f'Classes: {len(reference_dataset.classes)}')

## 5. Gardiner Sign List Database

In [ ]:
# Load Gardiner sign list from Dua-Khety database
# Source: https://github.com/MalakSadek/Dua-Khety

with open('Dua-Khety/database/GardinerCode.sql', 'r', errors='ignore') as f:
    content = f.read()

pattern = r"\('([^']+)',\s*0x[^,]+,\s*'([^']+)',\s*'([^']+)'"
matches = re.findall(pattern, content)

gardiner_db = {}
for match in matches:
    code = match[0]
    description = match[1]
    meaning = match[2]
    gardiner_db[code] = {
        'description': description,
        'meaning': meaning
    }

print(f'✅ Gardiner database loaded: {len(gardiner_db)} signs')
print('\nSample entries:')
for code in list(gardiner_db.keys())[:3]:
    print(f'  {code}: {gardiner_db[code]}')

## 6. Symbol Classification Pipeline

In [ ]:
def classify_symbol(img_crop):
    """
    Classify a single hieroglyphic symbol using the Siamese Network.
    Uses cosine distance comparison against reference embeddings.
    
    Args:
        img_crop: numpy array (BGR) or PIL Image of a single symbol
    Returns:
        (predicted_class, similarity_score)
    """
    if isinstance(img_crop, np.ndarray):
        img_crop = Image.fromarray(cv2.cvtColor(img_crop, cv2.COLOR_BGR2RGB))

    image_tensor = transform(img_crop).unsqueeze(0).to(device)

    with torch.no_grad():
        query_emb = model.embedding_net(image_tensor).cpu().squeeze().numpy()

    best_score = float('inf')
    predicted_class = None

    for item in embeddings_data:
        score = cosine(query_emb, item['embedding'])
        if score < best_score:
            best_score = score
            predicted_class = item['class_name']

    return predicted_class, 1 - best_score


def process_full_image(image_path):
    """
    Full pipeline: detect all symbols in an image and classify each one.
    
    Args:
        image_path: path to hieroglyphic image
    Returns:
        list of dicts with class, meaning, description, similarity, position
    """
    # 1. Load and preprocess
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.bitwise_not(gray)
    _, thresh = cv2.threshold(gray, 127, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 2. Find contours (symbol detection)
    contours, _ = cv2.findContours(thresh,
                                   cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)

    detected = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if 200 < area < 8000:
            x, y, w, h = cv2.boundingRect(cnt)
            if w < 150 and h < 150:
                detected.append((x, y, w, h))

    # 3. Classify each symbol
    results = []
    img_display = img.copy()

    for (x, y, w, h) in detected:
        symbol = img[y:y+h, x:x+w]
        cls, sim = classify_symbol(symbol)
        meaning = gardiner_db.get(cls, {}).get('meaning', 'unknown')
        description = gardiner_db.get(cls, {}).get('description', '')

        results.append({
            'class': cls,
            'meaning': meaning,
            'description': description,
            'similarity': sim,
            'position': (x, y, w, h)
        })

        # Draw bounding box and label
        cv2.rectangle(img_display, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(img_display, cls, (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

    # 4. Display results
    plt.figure(figsize=(15, 12))
    plt.imshow(cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB))
    plt.title(f'Detected {len(results)} hieroglyphic symbols', fontsize=14)
    plt.axis('off')
    plt.show()

    return results


print('✅ Classification pipeline ready!')

## 7. Translation with Gemini AI

In [ ]:
# Setup Gemini API
# Get your free API key from: https://aistudio.google.com
GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY'
genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel('gemini-1.5-flash')

print('Gemini API configured!')

In [ ]:
def translate_by_columns(results_sorted, img_width=1000):
    """
    Translate hieroglyphic text column by column using Gemini AI.
    Handles the ambiguity of reading direction by processing each column separately.
    
    Args:
        results_sorted: list of classified symbols sorted by position
        img_width: width of the original image (for column detection)
    Returns:
        (column_translations, overall_translation)
    """
    # Divide symbols into columns based on x position
    col_width = img_width // 10
    columns = {}
    for r in results_sorted:
        col_num = r['position'][0] // col_width
        if col_num not in columns:
            columns[col_num] = []
        columns[col_num].append(r)

    # Translate each column
    column_translations = []
    for col_num in sorted(columns.keys()):
        col_symbols = columns[col_num]
        symbols_text = ', '.join([f"{r['class']} ({r['description']})"
                                  for r in col_symbols])
        prompt = f'These hieroglyphs form one column: {symbols_text}. Translate briefly.'
        response = gemini.generate_content(prompt)
        col_translation = response.text
        column_translations.append(col_translation)
        print(f'Column {col_num+1}: {col_translation[:80]}...')

    # Generate overall meaning from all columns
    all_columns = '\n'.join([f'Column {i+1}: {t}'
                             for i, t in enumerate(column_translations)])

    summary_prompt = f"""
    These are translations of individual columns of an ancient Egyptian hieroglyphic text:
    {all_columns}

    Based on all columns together, what is the overall meaning of this text?
    Give a single coherent translation.
    """

    summary = gemini.generate_content(summary_prompt)

    print('\n' + '='*60)
    print('📜 FULL TRANSLATION:')
    print('='*60)
    print(summary.text)

    return column_translations, summary.text


print('Translation pipeline ready!')

## 8. Run Full Pipeline

In [ ]:


IMAGE_PATH = '/content/egyptianTexts3.jpg'  
# Step 1: Detect and classify all symbols
print('Step 1: Detecting and classifying symbols...')
results = process_full_image(IMAGE_PATH)

# Step 2: Sort symbols by position (top-to-bottom, left-to-right)
results_sorted = sorted(results,
                        key=lambda r: (r['position'][1]//50,
                                       r['position'][0]))

# Step 3: Print detected symbols
print(f'\nStep 2: Detected {len(results)} symbols')
print('-'*60)
for r in results_sorted[:20]:
    print(f"Symbol: {r['class']:5} | {r['description']:30} | Meaning: {r['meaning']:30} | Sim: {r['similarity']:.2f}")

# Step 4: Translate
print('\nStep 3: Translating...')
col_translations, overall = translate_by_columns(results_sorted)

## Limitations & Future Work

**Current Limitations:**
- OpenCV-based detection may miss some symbols or merge adjacent ones
- Translation accuracy depends on detection quality

**Future Improvements:**
- Use YOLOv8 for more accurate symbol detection
- Implement automatic reading direction detection based on symbol orientation
- Fine-tune a dedicated translation model on hieroglyphic texts